# Lab 08 — Benchmarking S3 Operations

Previous labs showed *how* S3 operations work. This lab measures *how fast* they are
and answers four practical questions:

1. **Single PUT vs. Multipart** — at what file size does multipart become faster?
2. **Part size** — what is the optimal part size for RustFS?
3. **Parallelism** — how does worker count scale throughput?
4. **Storage overhead** — how much extra space does EC use vs. theoretical replication?

All results are plotted with matplotlib so you can see the numbers at a glance.

---
## Setup

In [ ]:
import statistics
import sys
import time
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt

sys.path.insert(0, '../scripts')
from lab_utils import cleanup_bucket, ensure_bucket, get_s3_client

s3 = get_s3_client()
BUCKET = 'lab8-bench'
ensure_bucket(s3, BUCKET)

MB = 1024 * 1024


def make_payload(size_bytes: int) -> bytes:
    """Return a repeating-pattern payload of the requested size."""
    pattern = bytes(range(256))
    return (pattern * (size_bytes // 256 + 1))[:size_bytes]


def measure_put(size_bytes: int, key: str, runs: int = 3) -> float:
    """Return median throughput (MB/s) for single PUT."""
    payload = make_payload(size_bytes)
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        s3.put_object(Bucket=BUCKET, Key=key, Body=payload)
        times.append(time.perf_counter() - t0)
    s3.delete_object(Bucket=BUCKET, Key=key)
    return (size_bytes / MB) / statistics.median(times)


def measure_multipart(size_bytes: int, part_size: int, workers: int, key: str) -> float:
    """Return throughput (MB/s) for a multipart upload."""
    payload = make_payload(size_bytes)
    parts = [payload[i:i + part_size] for i in range(0, size_bytes, part_size)]

    resp = s3.create_multipart_upload(Bucket=BUCKET, Key=key)
    uid = resp['UploadId']

    completed = []
    t0 = time.perf_counter()

    def _upload(args):
        pnum, data = args
        r = s3.upload_part(Bucket=BUCKET, Key=key, UploadId=uid,
                           PartNumber=pnum, Body=data)
        return {'PartNumber': pnum, 'ETag': r['ETag']}

    with ThreadPoolExecutor(max_workers=workers) as ex:
        completed = list(ex.map(_upload, enumerate(parts, 1)))

    elapsed = time.perf_counter() - t0
    s3.complete_multipart_upload(
        Bucket=BUCKET, Key=key, UploadId=uid,
        MultipartUpload={'Parts': sorted(completed, key=lambda p: p['PartNumber'])},
    )
    s3.delete_object(Bucket=BUCKET, Key=key)
    return (size_bytes / MB) / elapsed


print('✅ Benchmark harness ready')

---
## Experiment 1 — Single PUT vs. Multipart Throughput

At what file size does switching to multipart start paying off?

In [ ]:
sizes_mb = [1, 5, 10, 25, 50, 100]
put_throughput = []
mp_throughput = []

for smb in sizes_mb:
    size = smb * MB
    tp_put = measure_put(size, f'bench/single_{smb}mb.bin')
    tp_mp  = measure_multipart(size, part_size=10 * MB, workers=4, key=f'bench/multi_{smb}mb.bin')
    put_throughput.append(tp_put)
    mp_throughput.append(tp_mp)
    print(f'  {smb:>4} MB  PUT={tp_put:5.1f} MB/s  Multipart={tp_mp:5.1f} MB/s')

print('\n✅ Experiment 1 done')

---
## Experiment 2 — Optimal Part Size

How does part size affect multipart throughput for a fixed 50 MB object?

In [ ]:
part_sizes_mb = [5, 8, 10, 16, 25, 50]
part_throughputs = []
FIXED_SIZE = 50 * MB

for psmb in part_sizes_mb:
    tp = measure_multipart(FIXED_SIZE, part_size=psmb * MB, workers=4,
                           key=f'bench/partsize_{psmb}mb.bin')
    part_throughputs.append(tp)
    print(f'  Part size {psmb:>3} MB → {tp:.1f} MB/s')

best_idx = part_throughputs.index(max(part_throughputs))
print(f'\n✅ Best part size: {part_sizes_mb[best_idx]} MB ({part_throughputs[best_idx]:.1f} MB/s)')

---
## Experiment 3 — Parallel Worker Scaling

How does the number of upload threads scale throughput for a 50 MB object?

In [ ]:
worker_counts = [1, 2, 4, 8]
worker_throughputs = []

for w in worker_counts:
    tp = measure_multipart(50 * MB, part_size=10 * MB, workers=w,
                           key=f'bench/workers_{w}.bin')
    worker_throughputs.append(tp)
    speedup = tp / worker_throughputs[0]
    print(f'  Workers={w}  throughput={tp:.1f} MB/s  speedup={speedup:.2f}×')

print('\n✅ Experiment 3 done')

---
## Results Dashboard

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('RustFS S3 Benchmark Results', fontsize=14, fontweight='bold')

# Plot 1: PUT vs Multipart
ax = axes[0]
ax.plot(sizes_mb, put_throughput, 'o-', label='Single PUT', color='#e74c3c')
ax.plot(sizes_mb, mp_throughput, 's-', label='Multipart (4 workers, 10 MB parts)', color='#2ecc71')
ax.set_xlabel('File size (MB)')
ax.set_ylabel('Throughput (MB/s)')
ax.set_title('PUT vs. Multipart Upload')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Part size sweep
ax = axes[1]
best_color = ['#e74c3c' if i == best_idx else '#3498db' for i in range(len(part_sizes_mb))]
ax.bar([str(p) for p in part_sizes_mb], part_throughputs, color=best_color)
ax.set_xlabel('Part size (MB)')
ax.set_ylabel('Throughput (MB/s)')
ax.set_title('Optimal Part Size (50 MB object)')
ax.text(0.5, 0.97, f'Best: {part_sizes_mb[best_idx]} MB parts',
        transform=ax.transAxes, ha='center', va='top', color='#e74c3c', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Plot 3: Worker scaling
ax = axes[2]
speedups = [t / worker_throughputs[0] for t in worker_throughputs]
ax.plot(worker_counts, speedups, 'D-', color='#9b59b6', label='Actual speedup')
ax.plot(worker_counts, worker_counts, '--', color='gray', alpha=0.5, label='Linear (ideal)')
ax.set_xlabel('Worker threads')
ax.set_ylabel('Speedup vs. 1 worker')
ax.set_title('Parallelism Scaling (50 MB, 10 MB parts)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Cleanup

In [ ]:
cleanup_bucket(s3, BUCKET)
print('✅ Cleanup complete')

---
## Summary

| Finding | Takeaway |
|---------|----------|
| PUT vs Multipart crossover | Multipart wins above ~10–25 MB (depends on network) |
| Optimal part size | ~10–16 MB for RustFS on localhost |
| Worker scaling | Near-linear up to 4 workers; diminishing returns beyond |

In production (remote object storage), multipart advantages become even more pronounced
because network latency is higher and parallelism hides it.
